In [ ]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
import cv2
from pynq import allocate
from pynq_dpu import DpuOverlay
import xir
import vart

In [ ]:
from pathlib import Path
import hashlib

DPU_BITSTREAM = "dpu.bit"

WINDOW_RADIUS = 3
N_STACK = 8
INPUT_HEIGHT, INPUT_WIDTH = 112, 96
SEG_LABELS = [2, 3, 4, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 26, 28]
OUTPUT_DIR = "./fpga_results"

QUIVER_AUTO_GAIN = True
QUIVER_TARGET_P95 = 2.0
QUIVER_MAX_GAIN = 20.0
QUIVER_DISPLAY_FLIP_Y = True


def resolve_file(default_path, extra_candidates=None):
    candidates = [Path(default_path)]
    for candidate in extra_candidates or []:
        candidates.append(Path(candidate))
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    return str(candidates[0])


XMODEL_PATH = resolve_file(
    './vxm_2p5d_pt_v3.xmodel',
    [
        '../../out/v2.5/vxm_2p5d_pt_v3/compiled/vxm_2p5d_pt_v3.xmodel',
        './out/v2.5/vxm_2p5d_pt_v3/compiled/vxm_2p5d_pt_v3.xmodel',
    ],
)
MR_VOLUME_PATH = resolve_file('./1BA001_mr.npy', ['./data/test_data/1BA001_mr.npy'])
CT_VOLUME_PATH = resolve_file('./1BA005_ct.npy', ['./data/test_data/1BA005_ct.npy'])
MR_SEG_PATH = resolve_file('./1BA001_mr_seg.npy', ['./data/test_data/1BA001_mr_seg.npy'])
CT_SEG_PATH = resolve_file('./1BA005_ct_seg.npy', ['./data/test_data/1BA005_ct_seg.npy'])

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Configuration:')
print(f'  XMODEL: {XMODEL_PATH}')
print('  Test data: CROSS-SUBJECT (1BA001 MR -> 1BA005 CT)')
print(f'  Input size: {INPUT_HEIGHT}x{INPUT_WIDTH}')
print(f'  Stack size: {N_STACK} slices')
print(f'  Output dir: {OUTPUT_DIR}')
print(f'  Quiver display: auto_gain={QUIVER_AUTO_GAIN}, target_p95={QUIVER_TARGET_P95}, max_gain={QUIVER_MAX_GAIN}, flip_y={QUIVER_DISPLAY_FLIP_Y}')


In [ ]:
overlay = DpuOverlay(DPU_BITSTREAM)
graph = xir.Graph.deserialize(XMODEL_PATH)
subgraphs = graph.get_root_subgraph().toposort_child_subgraph()

dpu_subgraph = None
for subgraph in subgraphs:
    if subgraph.get_attr('device') == "DPU":
        dpu_subgraph = subgraph
        break

if dpu_subgraph is None:
    raise RuntimeError("No DPU subgraph found")

dpu_runner = vart.Runner.create_runner(dpu_subgraph, "run")

In [ ]:
input_tensors = dpu_runner.get_input_tensors()
output_tensors = dpu_runner.get_output_tensors()

print("="*70)
print("QUANTIZATION SCALES")
print("="*70)

print(f"\nInput tensors: {len(input_tensors)}")
for i, tensor in enumerate(input_tensors):
    print(f"  {i}. {tensor.name}: {tensor.dims}")
    try:
        fix_point = tensor.get_attr('fix_point')
        scale = 2**fix_point
        print(f"     fix_point={fix_point}, scale={scale}")
    except Exception as e:
        print(f"     Error getting fix_point: {e}")

print(f"\nOutput tensors: {len(output_tensors)}")
for i, tensor in enumerate(output_tensors):
    print(f"  {i}. {tensor.name}: {tensor.dims}")
    try:
        fix_point = tensor.get_attr('fix_point')
        scale = 2**fix_point
        print(f"     fix_point={fix_point}, scale={scale}")
        print(f"     Dequantization: float_val = int8_val / {scale}")
    except Exception as e:
        print(f"     Error getting fix_point: {e}")

input_shape = tuple(input_tensors[0].dims)
output_shape = tuple(output_tensors[0].dims)

# CRITICAL: Map scales to the correct inputs based on tensor NAMES, not indices!
# The DPU expects inputs in a specific order based on the compiled model
FIXED_SCALE = None
MOVING_SCALE = None

for tensor in input_tensors:
    fix_point = tensor.get_attr('fix_point')
    scale = 2**fix_point
    if 'fixed' in tensor.name.lower():
        FIXED_SCALE = scale
    elif 'moving' in tensor.name.lower():
        MOVING_SCALE = scale

# Fallback if names don't contain 'fixed'/'moving'
if FIXED_SCALE is None or MOVING_SCALE is None:
    print("\nWarning: WARNING: Could not determine input order from tensor names!")
    print("   Using index-based assignment (may be incorrect)")
    FIXED_SCALE = 2 ** input_tensors[0].get_attr('fix_point')
    MOVING_SCALE = 2 ** input_tensors[1].get_attr('fix_point') if len(input_tensors) > 1 else FIXED_SCALE

try:
    OUTPUT_SCALE = 2 ** output_tensors[0].get_attr('fix_point')
except:
    OUTPUT_SCALE = None

print(f"\n" + "="*70)
print(f"SCALES TO USE IN INFERENCE:")
print(f"="*70)
print(f"  FIXED_SCALE = {FIXED_SCALE}")
print(f"  MOVING_SCALE = {MOVING_SCALE}")
print(f"  OUTPUT_SCALE = {OUTPUT_SCALE}")
print(f"="*70)

if OUTPUT_SCALE == 16:
    print(f"\nSuccess: OUTPUT_SCALE=16 detected (flow_scale=0.1x model)")
    print(f"   This is the FPGA-optimized model with integrated flow scaling")
elif OUTPUT_SCALE == 64:
    print(f"\nNOTE: OUTPUT_SCALE=64 detected")
elif OUTPUT_SCALE == 32:
    print(f"\nNOTE: OUTPUT_SCALE=32 detected")
elif OUTPUT_SCALE is not None:
    print(f"\nWARNING: Unexpected OUTPUT_SCALE={OUTPUT_SCALE}")

In [ ]:
def extract_slice_stack(volume, axis, z, window_radius, target_size=(112, 96)):
    wr = window_radius
    if axis == 0:
        stack = volume[z-wr:z+wr+1]
    elif axis == 1:
        stack = volume[:, z-wr:z+wr+1, :].transpose(1, 0, 2)
    else:
        stack = volume[:, :, z-wr:z+wr+1].transpose(2, 0, 1)

    resized_stack = []
    for i in range(stack.shape[0]):
        resized = cv2.resize(stack[i], (target_size[1], target_size[0]), interpolation=cv2.INTER_LINEAR)
        resized_stack.append(resized)

    return np.ascontiguousarray(np.array(resized_stack))


def pad_stack_to_n(stack, n_stack):
    if stack.shape[0] == n_stack:
        return stack
    if stack.shape[0] > n_stack:
        return stack[:n_stack]
    out = stack.copy()
    while out.shape[0] < n_stack:
        out = np.concatenate([out, out[-1:]], axis=0)
    return out


def apply_flow_2d(image, flow):
    h, w = image.shape

    if flow.shape[1] != h or flow.shape[2] != w:
        flow_x = cv2.resize(flow[0], (w, h), interpolation=cv2.INTER_LINEAR)
        flow_y = cv2.resize(flow[1], (w, h), interpolation=cv2.INTER_LINEAR)
        scale_x = w / flow.shape[2]
        scale_y = h / flow.shape[1]
        flow_x = flow_x * scale_x
        flow_y = flow_y * scale_y
    else:
        flow_x = flow[0]
        flow_y = flow[1]

    y_coords, x_coords = np.mgrid[0:h, 0:w].astype(np.float32)
    new_x = x_coords + flow_x
    new_y = y_coords + flow_y
    return cv2.remap(image.astype(np.float32), new_x, new_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)


def apply_flow_2d_nearest(image, flow):
    h, w = image.shape

    if flow.shape[1] != h or flow.shape[2] != w:
        flow_x = cv2.resize(flow[0], (w, h), interpolation=cv2.INTER_LINEAR)
        flow_y = cv2.resize(flow[1], (w, h), interpolation=cv2.INTER_LINEAR)
        scale_x = w / flow.shape[2]
        scale_y = h / flow.shape[1]
        flow_x = flow_x * scale_x
        flow_y = flow_y * scale_y
    else:
        flow_x = flow[0]
        flow_y = flow[1]

    y_coords, x_coords = np.mgrid[0:h, 0:w].astype(np.float32)
    new_x = x_coords + flow_x
    new_y = y_coords + flow_y
    warped = cv2.remap(image.astype(np.float32), new_x, new_y, cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT)
    return warped.astype(np.float32)


def _quiver_vis_gain(flow, target_p95=2.0, max_gain=20.0):
    mag = np.sqrt((flow[0] ** 2) + (flow[1] ** 2))
    p95 = float(np.percentile(mag, 95))
    if p95 <= 1e-8:
        return 1.0, p95
    gain = float(np.clip(target_p95 / p95, 1.0, max_gain))
    return gain, p95


def plot_quiver(ax, flow, background=None, step=4, auto_gain=True, target_p95=2.0, max_gain=20.0, display_flip_y=False):
    h, w = flow.shape[1], flow.shape[2]
    gain = 1.0
    p95 = 0.0
    if auto_gain:
        gain, p95 = _quiver_vis_gain(flow, target_p95=target_p95, max_gain=max_gain)

    y, x = np.mgrid[0:h:step, 0:w:step]
    fx = flow[0, ::step, ::step] * gain
    fy = flow[1, ::step, ::step] * gain
    if display_flip_y:
        fy = -fy
    bg = np.asarray(background) if background is not None else np.zeros((h, w), dtype=np.float32)
    ax.imshow(bg, cmap='gray', origin='upper')
    ax.quiver(x, y, fx, fy, color='red', angles='xy', scale_units='xy', scale=1, headwidth=3, headlength=4, alpha=0.8)
    ax.set_xlim(0, w - 1)
    ax.set_ylim(h - 1, 0)
    return gain, p95


def compute_dice(pred_seg, gt_seg, labels=SEG_LABELS):
    dice_scores = []
    for lbl in labels:
        pred_mask = (pred_seg == lbl).astype(np.float32)
        gt_mask = (gt_seg == lbl).astype(np.float32)
        gt_sum = gt_mask.sum()
        if gt_sum == 0:
            continue
        intersection = (pred_mask * gt_mask).sum()
        union = pred_mask.sum() + gt_sum
        dice_scores.append((2.0 * intersection + 1e-5) / (union + 1e-5))
    return float(np.mean(dice_scores)) if dice_scores else 0.0


def _run_int8_dpu(inputs):
    input_bufs = []
    output_buf = None
    try:
        for arr in inputs:
            buf = allocate(shape=tuple(arr.shape), dtype=np.int8)
            np.copyto(buf, np.ascontiguousarray(arr, dtype=np.int8))
            if hasattr(buf, 'sync_to_device'):
                buf.sync_to_device()
            input_bufs.append(buf)

        output_buf = allocate(shape=tuple(output_shape), dtype=np.int8)
        job_id = dpu_runner.execute_async(input_bufs, [output_buf])
        dpu_runner.wait(job_id)
        if hasattr(output_buf, 'sync_from_device'):
            output_buf.sync_from_device()
        return np.array(output_buf, copy=True)
    finally:
        for buf in input_bufs:
            if hasattr(buf, 'freebuffer'):
                buf.freebuffer()
        if output_buf is not None and hasattr(output_buf, 'freebuffer'):
            output_buf.freebuffer()


def run_dpu_inference(moving_stack, fixed_stack):
    moving_stack = pad_stack_to_n(moving_stack, N_STACK)
    fixed_stack = pad_stack_to_n(fixed_stack, N_STACK)

    moving_input = moving_stack.transpose(1, 2, 0)[np.newaxis, ...].astype(np.float32)
    fixed_input = fixed_stack.transpose(1, 2, 0)[np.newaxis, ...].astype(np.float32)

    if len(input_tensors) == 2:
        if FIXED_SCALE is None or MOVING_SCALE is None:
            raise RuntimeError('FIXED_SCALE and MOVING_SCALE are required for a two-input model')

        moving_input_q = np.clip(np.round(moving_input * MOVING_SCALE), -128, 127).astype(np.int8)
        fixed_input_q = np.clip(np.round(fixed_input * FIXED_SCALE), -128, 127).astype(np.int8)

        first_tensor_name = input_tensors[0].name.lower()
        if 'fixed' in first_tensor_name:
            inputs = [fixed_input_q, moving_input_q]
        elif 'moving' in first_tensor_name:
            inputs = [moving_input_q, fixed_input_q]
        else:
            print('Warning: could not infer tensor order from names, using [fixed, moving]')
            inputs = [fixed_input_q, moving_input_q]
    else:
        combined = np.concatenate([moving_stack, fixed_stack], axis=0).astype(np.float32)
        input_data = combined.transpose(1, 2, 0)[np.newaxis, ...]
        expected_ch = N_STACK * 2
        if input_data.shape[-1] != expected_ch:
            raise RuntimeError(f'Input channel mismatch: got {input_data.shape[-1]}, expected {expected_ch}')
        input_scale = FIXED_SCALE if FIXED_SCALE else (2 ** input_tensors[0].get_attr('fix_point'))
        inputs = [np.clip(np.round(input_data * input_scale), -128, 127).astype(np.int8)]

    start = time.time()
    output_data = _run_int8_dpu(inputs)
    duration = time.time() - start

    if OUTPUT_SCALE is None:
        raise RuntimeError('OUTPUT_SCALE not set')

    flow = output_data.astype(np.float32) / OUTPUT_SCALE
    if len(flow.shape) == 4:
        flow = flow[0]
        if flow.shape[-1] == 2:
            flow = flow.transpose(2, 0, 1)

    return flow.astype(np.float32), duration


In [ ]:
moving_vol = np.load(MR_VOLUME_PATH).astype(np.float32)
fixed_vol = np.load(CT_VOLUME_PATH).astype(np.float32)

try:
    moving_seg = np.load(MR_SEG_PATH).astype(np.float32)
    fixed_seg = np.load(CT_SEG_PATH).astype(np.float32)
    has_seg = True
except:
    has_seg = False
    moving_seg = np.zeros_like(moving_vol)
    fixed_seg = np.zeros_like(fixed_vol)
    print("Segmentation maps not found, skipping Dice evaluation.")

print(f"Moving: {moving_vol.shape}, range [{moving_vol.min():.3f}, {moving_vol.max():.3f}]")
print(f"Fixed: {fixed_vol.shape}, range [{fixed_vol.min():.3f}, {fixed_vol.max():.3f}]")
if has_seg:
    print(f"Moving Seg: {moving_seg.shape}, labels: {np.unique(moving_seg)}")
    print(f"Fixed Seg: {fixed_seg.shape}, labels: {np.unique(fixed_seg)}")


In [ ]:
z_center = moving_vol.shape[0] // 2
moving_stack = extract_slice_stack(moving_vol, 0, z_center, WINDOW_RADIUS)
fixed_stack = extract_slice_stack(fixed_vol, 0, z_center, WINDOW_RADIUS)
if has_seg:
    moving_seg_center_full = moving_seg[z_center]
    fixed_seg_center_full = fixed_seg[z_center]
    moving_seg_center = cv2.resize(moving_seg_center_full, (INPUT_WIDTH, INPUT_HEIGHT), interpolation=cv2.INTER_NEAREST).astype(np.int16)
    fixed_seg_center = cv2.resize(fixed_seg_center_full, (INPUT_WIDTH, INPUT_HEIGHT), interpolation=cv2.INTER_NEAREST).astype(np.int16)
else:
    moving_seg_center_full = None
    fixed_seg_center_full = None
    moving_seg_center = None
    fixed_seg_center = None

flow, inf_time = run_dpu_inference(moving_stack, fixed_stack)

print("\n" + "="*70)
print("INFERENCE VERIFICATION")
print("="*70)
print(f"Inference time: {inf_time*1000:.2f} ms")
print(f"Flow shape: {flow.shape}")
print(f"Flow range: [{flow.min():.4f}, {flow.max():.4f}]")
print(f"Flow mean: {flow.mean():.4f}, std: {flow.std():.4f}")

if abs(flow.max()) < 0.1 and abs(flow.min()) < 0.1:
    print("\nERROR: Flow values are near zero!")
    print("This indicates dequantization is not working correctly.")
    print(f"Check that OUTPUT_SCALE={OUTPUT_SCALE} is being applied.")
elif abs(flow.max()) > 0.5 or abs(flow.min()) > 0.5:
    print("\nFlow values look reasonable (not near zero)")
    print("Dequantization appears to be working correctly.")
else:
    print("\nFlow values are small but non-zero")
    print("May need to verify quantization configuration.")
print("="*70 + "\n")

center_idx = WINDOW_RADIUS
moving_center = moving_stack[center_idx]
fixed_center = fixed_stack[center_idx]
warped = apply_flow_2d(moving_center, flow)

if has_seg:
    warped_seg = apply_flow_2d_nearest(moving_seg_center, flow).astype(np.int16)
    dice_before = compute_dice(moving_seg_center, fixed_seg_center)
    dice_after = compute_dice(warped_seg, fixed_seg_center)
else:
    warped_seg = np.zeros_like(moving_center)
    dice_before = 0.0
    dice_after = 0.0

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(moving_center, cmap='gray')
axes[0, 0].set_title('Moving')
axes[0, 0].axis('off')

axes[0, 1].imshow(fixed_center, cmap='gray')
axes[0, 1].set_title('Fixed')
axes[0, 1].axis('off')

axes[0, 2].imshow(warped, cmap='gray')
axes[0, 2].set_title('Warped')
axes[0, 2].axis('off')

q_gain, q_p95 = plot_quiver(axes[1, 0], flow, background=moving_center, step=4, auto_gain=QUIVER_AUTO_GAIN, target_p95=QUIVER_TARGET_P95, max_gain=QUIVER_MAX_GAIN, display_flip_y=QUIVER_DISPLAY_FLIP_Y)
axes[1, 0].set_title(f'Flow Details (Quiver x{q_gain:.1f}, flip_y={QUIVER_DISPLAY_FLIP_Y})')
axes[1, 0].axis('off')

if has_seg:
    axes[1, 1].imshow(moving_seg_center, cmap='tab20', alpha=0.5)
    axes[1, 1].set_title(f'Seg Before (Dice: {dice_before:.4f})')
else:
    diff_before = np.abs(moving_center - fixed_center)
    axes[1, 1].imshow(diff_before, cmap='hot')
    axes[1, 1].set_title(f'Diff Before (MAE: {diff_before.mean():.4f})')
axes[1, 1].axis('off')

if has_seg:
    axes[1, 2].imshow(warped_seg, cmap='tab20', alpha=0.5)
    axes[1, 2].set_title(f'Seg After (Dice: {dice_after:.4f})')
else:
    diff_after = np.abs(warped - fixed_center)
    axes[1, 2].imshow(diff_after, cmap='hot')
    axes[1, 2].set_title(f'Diff After (MAE: {diff_after.mean():.4f})')
axes[1, 2].axis('off')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/single_slice.png", dpi=150)
plt.show()


In [ ]:
def infer_full_volume(moving_vol, fixed_vol, moving_seg=None, axis=0):
    D = moving_vol.shape[axis]
    wr = WINDOW_RADIUS
    
    if axis == 0:
        orig_h, orig_w = moving_vol.shape[1], moving_vol.shape[2]
    elif axis == 1:
        orig_h, orig_w = moving_vol.shape[0], moving_vol.shape[2]
    else:
        orig_h, orig_w = moving_vol.shape[0], moving_vol.shape[1]
    
    warped_volume = np.zeros_like(moving_vol)
    if moving_seg is not None:
        warped_seg_volume = np.zeros_like(moving_seg)
    else:
        warped_seg_volume = None
        
    flow_volume = []
    inference_times = []
    
    start_total = time.time()
    
    for z in range(wr, D - wr):
        moving_stack = extract_slice_stack(moving_vol, axis, z, wr)
        fixed_stack = extract_slice_stack(fixed_vol, axis, z, wr)
        
        flow, inf_time = run_dpu_inference(moving_stack, fixed_stack)
        inference_times.append(inf_time)
        flow_volume.append(flow)
        
        if axis == 0:
            moving_center = moving_vol[z]
            if moving_seg is not None: moving_seg_center = moving_seg[z]
        elif axis == 1:
            moving_center = moving_vol[:, z, :]
            if moving_seg is not None: moving_seg_center = moving_seg[:, z, :]
        else:
            moving_center = moving_vol[:, :, z]
            if moving_seg is not None: moving_seg_center = moving_seg[:, :, z]
        
        warped_slice = apply_flow_2d(moving_center, flow)
        if moving_seg is not None:
            warped_seg_slice = apply_flow_2d_nearest(moving_seg_center, flow)
        
        if axis == 0:
            warped_volume[z] = warped_slice
            if moving_seg is not None: warped_seg_volume[z] = warped_seg_slice
        elif axis == 1:
            warped_volume[:, z, :] = warped_slice
            if moving_seg is not None: warped_seg_volume[:, z, :] = warped_seg_slice
        else:
            warped_volume[:, :, z] = warped_slice
            if moving_seg is not None: warped_seg_volume[:, :, z] = warped_seg_slice
        
        if (z - wr + 1) % 10 == 0:
            print(f"Processed {z - wr + 1}/{D - 2*wr} (avg {np.mean(inference_times)*1000:.2f} ms/slice)")
    
    total_time = time.time() - start_total
    
    print(f"Total time: {total_time:.2f} s")
    print(f"Average: {np.mean(inference_times)*1000:.2f} ms/slice")
    print(f"Throughput: {(D - 2*wr) / total_time:.2f} slices/s")
    
    return warped_volume, np.array(flow_volume), inference_times, warped_seg_volume

warped_vol, flow_vol, inf_times, warped_seg_vol = infer_full_volume(moving_vol, fixed_vol, moving_seg if has_seg else None, axis=0)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(np.array(inf_times) * 1000, bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(np.mean(inf_times) * 1000, color='red', linestyle='--', label=f'Mean: {np.mean(inf_times)*1000:.2f} ms')
axes[0].set_xlabel('Inference Time (ms)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Inference Time Distribution')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(np.array(inf_times) * 1000, marker='o', markersize=3, alpha=0.6)
axes[1].axhline(np.mean(inf_times) * 1000, color='red', linestyle='--', label=f'Mean: {np.mean(inf_times)*1000:.2f} ms')
axes[1].set_xlabel('Slice Index')
axes[1].set_ylabel('Inference Time (ms)')
axes[1].set_title('Inference Time per Slice')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/performance.png", dpi=150)
plt.show()

print(f"Slices: {len(inf_times)}")
print(f"Mean: {np.mean(inf_times)*1000:.2f} ms")
print(f"Std: {np.std(inf_times)*1000:.2f} ms")
print(f"Min: {np.min(inf_times)*1000:.2f} ms")
print(f"Max: {np.max(inf_times)*1000:.2f} ms")

In [ ]:
slice_indices = [moving_vol.shape[0]//4, moving_vol.shape[0]//2, 3*moving_vol.shape[0]//4]

fig, axes = plt.subplots(len(slice_indices), 4, figsize=(16, 4*len(slice_indices)))

for i, z in enumerate(slice_indices):
    if WINDOW_RADIUS <= z < moving_vol.shape[0] - WINDOW_RADIUS:
        axes[i, 0].imshow(moving_vol[z], cmap='gray')
        axes[i, 0].set_title(f'Moving - Slice {z}')
        axes[i, 0].axis('off')
        
        axes[i, 1].imshow(fixed_vol[z], cmap='gray')
        axes[i, 1].set_title(f'Fixed - Slice {z}')
        axes[i, 1].axis('off')
        
        axes[i, 2].imshow(warped_vol[z], cmap='gray')
        axes[i, 2].set_title(f'Warped - Slice {z}')
        axes[i, 2].axis('off')
        
        if has_seg:
            vol_z_flow = flow_vol[z - WINDOW_RADIUS]
            dice_score = compute_dice(warped_seg_vol[z], fixed_seg[z])
            q_gain, q_p95 = plot_quiver(
                axes[i, 3],
                vol_z_flow,
                background=moving_vol[z],
                step=4,
                auto_gain=QUIVER_AUTO_GAIN,
                target_p95=QUIVER_TARGET_P95,
                max_gain=QUIVER_MAX_GAIN,
                display_flip_y=QUIVER_DISPLAY_FLIP_Y,
            )
            axes[i, 3].set_title(f'Flow x{q_gain:.1f} (Dice: {dice_score:.4f})')
            axes[i, 3].axis('off')
        else:
            diff = np.abs(warped_vol[z] - fixed_vol[z])
            im = axes[i, 3].imshow(diff, cmap='hot')
            axes[i, 3].set_title(f'Diff (MAE: {diff.mean():.4f})')
            axes[i, 3].axis('off')
            plt.colorbar(im, ax=axes[i, 3])

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/volume_results.png", dpi=150)
plt.show()


In [ ]:
np.save(f"{OUTPUT_DIR}/warped_volume.npy", warped_vol)
np.save(f"{OUTPUT_DIR}/flow_volume.npy", flow_vol)

import json


def _stats(arr):
    arr = np.asarray(arr)
    return {
        'shape': list(arr.shape),
        'min': float(np.min(arr)),
        'max': float(np.max(arr)),
        'mean': float(np.mean(arr)),
        'std': float(np.std(arr)),
    }


def _md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def _mi_np(i, j, bins=32):
    i = np.asarray(i, dtype=np.float32)
    j = np.asarray(j, dtype=np.float32)
    i_norm = (i - i.min()) / (i.max() - i.min() + 1e-6)
    j_norm = (j - j.min()) / (j.max() - j.min() + 1e-6)
    i_idx = (i_norm * (bins - 1)).astype(np.int32)
    j_idx = (j_norm * (bins - 1)).astype(np.int32)
    hist, _, _ = np.histogram2d(i_idx.ravel(), j_idx.ravel(), bins=bins, range=[[0, bins - 1], [0, bins - 1]])
    pxy = hist / (np.sum(hist) + 1e-9)
    px = np.sum(pxy, axis=1, keepdims=True)
    py = np.sum(pxy, axis=0, keepdims=True)
    pxpy = px * py
    nz = pxy > 0
    return float(np.sum(pxy[nz] * np.log((pxy[nz] / (pxpy[nz] + 1e-9)) + 1e-9)))


results = {
    'bitstream': DPU_BITSTREAM,
    'xmodel_path': XMODEL_PATH,
    'xmodel_md5': _md5(XMODEL_PATH),
    'input_hw': [int(INPUT_HEIGHT), int(INPUT_WIDTH)],
    'n_stack': int(N_STACK),
    'total_slices': len(inf_times),
    'mean_time_ms': float(np.mean(inf_times) * 1000),
    'std_time_ms': float(np.std(inf_times) * 1000),
    'throughput_slices_per_sec': float(len(inf_times) / sum(inf_times)),
    'input_scale_fixed': float(FIXED_SCALE) if FIXED_SCALE else None,
    'input_scale_moving': float(MOVING_SCALE) if MOVING_SCALE else None,
    'output_scale': float(OUTPUT_SCALE) if OUTPUT_SCALE else None,
    'flow': _stats(flow),
    'dice_before': float(dice_before),
    'dice_after': float(dice_after),
    'mi_before': _mi_np(moving_center, fixed_center),
    'mi_after': _mi_np(warped, fixed_center),
}

with open(f"{OUTPUT_DIR}/results.json", 'w') as f:
    json.dump(results, f, indent=2)

print(f'Results saved to {OUTPUT_DIR}/results.json')


In [ ]:
del dpu_runner
del graph
print("Done")